# Color Gradient LB — Droplet Test

Two-component color gradient lattice Boltzmann simulation of a spherical droplet.
Component **a** is the solvent, component **b** is the droplet.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

import espressomd
import espressomd.lb

In [ ]:
DOMAIN_SIZE = 24
AGRID = 1.0
TAU = 0.01
RHO_0 = 1.0
EPSILON = 1e-6
RADIUS = 5.0
SMOOTHING_WIDTH = 2.0

# Desired LB viscosity: nu_LB = 1/6 gives omega = 1.0
# ESPResSo converts: nu_LB = nu_MD * tau / agrid^2
# So nu_MD = nu_LB / (tau / agrid^2)
VISCOSITY_LB = 1.0 / 6.0
VISCOSITY_MD = VISCOSITY_LB / (TAU / AGRID**2)

N_FRAMES = 200
STEPS_PER_FRAME = 1


def tanh_interpolation(distances, radius, smoothing_width, rho_outer, rho_inner):
    """Smooth tanh interface profile."""
    return (rho_outer-rho_inner) /2.0 *(np.tanh((distances-radius) /smoothing_width *2.64665) - 1.0) + rho_outer


def make_droplet(grid_size, radius, smoothing_width, rho_0, epsilon):
    """Generate initial density fields for a spherical droplet."""
    x = np.arange(grid_size) + 0.5
    xx, yy, zz = np.meshgrid(x, x, x, indexing='ij')
    center = grid_size / 2.0

    # Sphere (Euclidean distance)
    #distances = np.sqrt((xx - center)**2 + (yy - center)**2 + (zz - center)**2)
    
    # Square (Chebyshev distance) — uncomment to use instead
    distances = np.maximum(np.maximum(np.abs(xx - center), np.abs(yy - center)), np.abs(zz - center))

    rho_a = tanh_interpolation(distances, radius, smoothing_width,
                               rho_0, epsilon * rho_0)
    rho_b = tanh_interpolation(distances, radius, smoothing_width,
                               epsilon * rho_0, rho_0)
    return rho_a, rho_b


def read_densities(lbf, grid_size):
    """Read per-node densities into two 3D arrays."""
    rho_a = np.zeros((grid_size, grid_size, grid_size))
    rho_b = np.zeros((grid_size, grid_size, grid_size))
    for x in range(grid_size):
        for y in range(grid_size):
            for z in range(grid_size):
                d = lbf[x, y, z].density
                rho_a[x, y, z] = d[0]
                rho_b[x, y, z] = d[1]
    return rho_a, rho_b


def plot_densities(rho_a, rho_b, title=''):
    """Plot a z-midplane slice of both density fields."""
    z_mid = rho_a.shape[2] // 2
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    im1 = ax1.imshow(rho_a[:, :, z_mid].T, origin='lower', cmap='Blues', vmin=0, vmax=RHO_0)
    ax1.set_title('rho_a (solvent)')
    plt.colorbar(im1, ax=ax1)
    im2 = ax2.imshow(rho_b[:, :, z_mid].T, origin='lower', cmap='Reds', vmin=0, vmax=RHO_0)
    ax2.set_title('rho_b (droplet)')
    plt.colorbar(im2, ax=ax2)
    if title:
        fig.suptitle(title)
    plt.tight_layout()
    plt.show()

## Setup and initial state

In [ ]:
system = espressomd.System(box_l=[DOMAIN_SIZE * AGRID] * 3)
system.time_step = TAU
system.cell_system.skin = 0.4

lbf = espressomd.lb.LBFluid(
    agrid=AGRID, density=RHO_0, tau=TAU, kinematic_viscosity=[VISCOSITY_MD, VISCOSITY_MD])
system.lb = lbf

rho_a_init, rho_b_init = make_droplet(DOMAIN_SIZE, RADIUS, SMOOTHING_WIDTH, RHO_0, EPSILON)

for x in range(DOMAIN_SIZE):
    for y in range(DOMAIN_SIZE):
        for z in range(DOMAIN_SIZE):
            lbf[x, y, z].density = [rho_a_init[x, y, z], rho_b_init[x, y, z]]
lbf.init_two_component()

rho_a, rho_b = read_densities(lbf, DOMAIN_SIZE)
plot_densities(rho_a, rho_b, title='Initial state (t = 0)')

## Run simulation

In [ ]:
z_slice = DOMAIN_SIZE // 2
rho_a_frames = [rho_a[:, :, z_slice].copy()]
rho_b_frames = [rho_b[:, :, z_slice].copy()]

total_rho_a_init = np.sum(rho_a)
total_rho_b_init = np.sum(rho_b)
times = [0]
mass_a = [total_rho_a_init]
mass_b = [total_rho_b_init]

print(f'Running {N_FRAMES} frames x {STEPS_PER_FRAME} steps '
      f'= {N_FRAMES * STEPS_PER_FRAME} total steps ...')
for i in range(N_FRAMES):
    system.integrator.run(STEPS_PER_FRAME)
    rho_a, rho_b = read_densities(lbf, DOMAIN_SIZE)

    t = (i + 1) * STEPS_PER_FRAME
    times.append(t)
    mass_a.append(np.sum(rho_a))
    mass_b.append(np.sum(rho_b))

    rho_a_frames.append(rho_a[:, :, z_slice].copy())
    rho_b_frames.append(rho_b[:, :, z_slice].copy())

    if (i + 1) % 25 == 0:
        print(f'  t = {t:5d}:  rho_a = {mass_a[-1]:.8f},  '
              f'rho_b = {mass_b[-1]:.8f}')

print('Done.')

## Results

In [ ]:
# Final state
plot_densities(rho_a, rho_b, title=f'Final state (t = {times[-1]})')

# Mass conservation
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(times, np.array(mass_a) - total_rho_a_init, 'b-o', markersize=2, label='rho_a drift')
ax.plot(times, np.array(mass_b) - total_rho_b_init, 'r-s', markersize=2, label='rho_b drift')
ax.set_xlabel('Time step')
ax.set_ylabel('Total mass drift')
ax.set_title('Mass conservation')
ax.legend()
ax.axhline(0, color='k', linewidth=0.5)
plt.tight_layout()
plt.show()
print(f'Max |drift| rho_a: '
      f'{np.max(np.abs(np.array(mass_a) - total_rho_a_init)):.2e}')
print(f'Max |drift| rho_b: '
      f'{np.max(np.abs(np.array(mass_b) - total_rho_b_init)):.2e}')

# Animation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
im1 = ax1.imshow(rho_a_frames[0].T, origin='lower', cmap='Blues', vmin=0, vmax=RHO_0)
ax1.set_title('rho_a (solvent)')
plt.colorbar(im1, ax=ax1)
im2 = ax2.imshow(rho_b_frames[0].T, origin='lower', cmap='Reds', vmin=0, vmax=RHO_0)
ax2.set_title('rho_b (droplet)')
plt.colorbar(im2, ax=ax2)
suptitle = fig.suptitle('t = 0')
plt.tight_layout()


def update(frame):
    im1.set_data(rho_a_frames[frame].T)
    im2.set_data(rho_b_frames[frame].T)
    suptitle.set_text(f't = {frame * STEPS_PER_FRAME}')
    return im1, im2, suptitle


anim = FuncAnimation(fig, update, frames=len(rho_a_frames), interval=50, blit=True)
plt.close(fig)
HTML(anim.to_jshtml())